In [1]:
from nichenetpy.utils import (
    read_csv_cols
)
from nichenetpy.parameter_optimization import (
    construct_and_evaluate
)

from optuna import create_study
from optuna.trial import Trial
from optuna.samplers import TPESampler
from itertools import chain

import os
import requests
import pandas as pd
import session_info
import json
import numpy as np

c:\Users\victorm\Documents\nichenetpy\.hatch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
network_path = os.path.normpath("./tutorial_files/model_construction/human")
if not os.path.exists(network_path):
    os.makedirs(network_path)
for filename in (
    "gr_human.csv",
    "lr_network_human.csv",
    "lr_sig_human.csv",
    "optimized_source_weights.csv",
    "annotation_data_sources.csv"
):
    file_path = os.path.join(network_path, filename)
    if not os.path.exists(file_path):
        res = requests.get(f"https://zenodo.org/records/14929618/files/{filename}")
        with open(file_path, "wb") as file:
            file.write(res.content)

In [3]:
gr_network = pd.DataFrame(read_csv_cols(os.path.join(network_path, "gr_human.csv")))
lr_network = pd.DataFrame(read_csv_cols(os.path.join(network_path, "lr_network_human.csv")))
sig_network = pd.DataFrame(read_csv_cols(os.path.join(network_path, "lr_sig_human.csv")))

In [4]:
train_path = "D:/Data/nichenetpy/model_optimization"
with open(os.path.join(train_path, "settings_training_f1234.json"), "rb") as file:
    settings_CV = json.loads(file.read())
settings = settings_CV["settings"]

In [5]:
gr_network = gr_network[
    ((gr_network["database"] == "NicheNet_LT") & np.array([fr not in settings_CV["forbidden_ligands_nichenet"] for fr in gr_network["from"]]))
    |
    ((gr_network["database"] == "CytoSig") & np.array([fr not in settings_CV["forbidden_ligands_cytosig"] for fr in gr_network["from"]]))
]

In [6]:
source_names = sorted(set(chain(gr_network["source"], lr_network["source"], sig_network["source"])))

def objective(trial:Trial):
    source_weights = dict(
        (
            source_name,
            trial.suggest_float(
                name=source_name,
                low=0, # TODO
                high=1 # TODO
            )
        ) for source_name in source_names
    )
    res = construct_and_evaluate(
        source_weights,
        lr_network,
        gr_network,
        sig_network,
        settings
    )
    return (res[1], res[2])

study = create_study(
    sampler=TPESampler(),
    directions=["maximize", "maximize"]
)
study.optimize(
    objective,
    n_trials=1,
    n_jobs=-1
)

[I 2025-07-02 14:54:08,587] A new study created in memory with name: no-name-23a67880-d114-4317-be8c-947fd19683d5
c:\Users\victorm\Documents\nichenetpy\.hatch\Lib\site-packages\sklearn\metrics\_ranking.py:1030: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
c:\Users\victorm\Documents\nichenetpy\.hatch\Lib\site-packages\sklearn\metrics\_ranking.py:1183: UndefinedMetricWarning: No positive samples in y_true, true positive value should be meaningless
  warnings.warn(
C:\Users\victorm\Documents\nichenetpy\src\nichenetpy\metrics.py:161: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  pcc = pearsonr(response, prediction).statistic
c:\Users\victorm\Documents\nichenetpy\.hatch\Lib\site-packages\sklearn\metrics\_ranking.py:1030: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
c:\Users\victorm\Documents\nichenetpy\.hatch\Lib\site-pac

In [7]:
session_info.show()